# Tool catalog: browser media foundation

Run this notebook cell by cell in a disposable JupyterLab project with its live Python kernel at the server root. The generated red PNG is an actual local input, not a camera image. Each browser call returns a receipt immediately; wait for the media row to finish and run its inspection cell on a later turn. The AI questions use the selected ChatGPT connection and ask for real tool calls. They do not request device access.


## Make an exact local media reference

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
from PIL import Image

server_root = Path.cwd().resolve()  # Start JupyterLab with this disposable project as its server root.
source = server_root / f'browser-media-source-{uuid4().hex[:8]}.png'
Image.new('RGB', (2, 2), 'red').save(source)
ai_save_to = f'browser-media-ai-{uuid4().hex[:8]}.png'
source_ref = {'path': source.name, 'sha256': hashlib.sha256(source.read_bytes()).hexdigest()}
source_ref

## browser_capabilities

In [ ]:
from nbinlineai.tools import browser_capabilities
capabilities = browser_capabilities()
capabilities

In [ ]:
print('Capability receipt:', capabilities.status, capabilities.error)
if capabilities.status == 'completed':
    print('Browser capabilities:', capabilities.result)


Use &`browser_capabilities` now. Report the actual file-save capability and one concrete limit from its result; give the arguments you used. Do not request camera, microphone, or screen access.


## save_media

In [ ]:
from nbinlineai.tools import save_media
saved = save_media(source_ref, save_to='auto')
saved

In [ ]:
print('Save receipt:', saved.status, saved.error)
if saved.status == 'completed':
    print('Saved descriptor:', {key: saved.media.get(key) for key in ('media_id', 'path', 'sha256', 'mime_type')})
    print('File exists:', (server_root / saved.media['path']).is_file())
if saved.status == 'completed':
    saved_operation_id = saved.operation_id
    saved_media_id = saved.media['media_id']


Use &`save_media` on the exact disposable PNG in $`source_ref`, setting `save_to` to $`ai_save_to`. Wait for completion and report its generated path, SHA-256, and managed media ID so a later question can release precisely that media.


## operation_status

In [ ]:
from nbinlineai.tools import operation_status
assert saved.status == 'completed', 'Wait for the save receipt, then rerun this cell.'
lookup = operation_status(saved.operation_id)
lookup

In [ ]:
print('Status lookup:', lookup.status, lookup.error)
if lookup.status == 'completed':
    print('Original operation snapshot:', lookup.result)


Use &`operation_status` with operation ID $`saved_operation_id` for the completed direct save. Report its actual state and the ID you looked up.


## cancel_operation

In [ ]:
from nbinlineai.tools import cancel_operation
cancelled = cancel_operation(saved.operation_id)
cancelled

In [ ]:
print('Cancel after completed save:', cancelled.status, cancelled.error)
if cancelled.status == 'completed':
    print('Saved file still exists:', (server_root / saved.media['path']).is_file())


Use &`cancel_operation` with operation ID $`saved_operation_id` after that save completed. Report the actual response and whether the saved file remains; do not say this deletes a file.


## release_media

In [ ]:
picture = Image.open(server_root / saved.media['path'])
picture.load()
from nbinlineai.tools import release_media
released = release_media(saved.media['media_id'])
released

In [ ]:
from IPython.display import display
print('Release receipt:', released.status, released.error)
print('Saved file still exists:', (server_root / saved.media['path']).is_file())
print('Delivered Pillow image:', picture.size)
display(picture.resize((80, 80)))


Use &`release_media` on the media ID returned by your earlier `save_media` tool call, if that call produced a managed descriptor. Report the actual release status. Do not release or guess a different ID.


In [ ]:
saved_path = server_root / saved.media['path']
assert saved_path.exists() and picture.size == (2, 2)
saved_path.unlink()  # Remove only the exact file this example saved.
source.unlink(missing_ok=True)
(server_root / ai_save_to).unlink(missing_ok=True)  # Only the exact extra AI-save destination.
not saved_path.exists() and not source.exists()